- Miguel Baños Baladrón
- Fiz Garrido Escudero

La variabilidad del dataset no es solo barco o no barco. Hay tierra, agua, niebla, lluvia, sol. Dividir el dataset por estratificación de etiquetas de ship/no ship simplemente ayuda a tener balanceada las clases. Pero si entreno con 40 imagenes soleadas y 10 de lluvia y en test tengo 90 de lluvia, por mucha estratificación no sabria diferenciar correctamente unbarco en cualquier clima. Además, no tiene sentido clasificador atracado o no atracado si no hay barco.

Primero probar normal y ver si es muy mejorable o no

Probar todo esto, no tiene porque mejorar la metrica, tambien depende del reparto. Evitamos echarlo a suertes con cross-validation, mejor evaluación pero no mejor resultado

No todas las imágenes están en el csv de atracado.

# Práctica 1: Clasificación

### Importaciones

In [ ]:
import os
import glob
import torch
import pandas as pd
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

from torch.utils import data
from torch.utils.data import Subset
from torch.utils.data import DataLoader
from torchvision.io import read_image
from torchvision import transforms
from sklearn.model_selection import train_test_split

# 1.- Creación de Dataset personalizado

In [ ]:
class Dataset(data.Dataset):
    def __init__(self, image_path, ship_csv, docked_csv, transform=None):
        super().__init__()
        # Guardamos la ruta de cada imagen
        self.image_files = glob.glob(os.path.join(image_path, '*.jpg')) + \
                           glob.glob(os.path.join(image_path, '*.png'))

        # Leer csv
        ship = pd.read_csv(ship_csv, header=None)
        docked = pd.read_csv(docked_csv, header=None)

        # Diccionario de etiquetas
        self.labels = {}

        for i in range(len(ship)):
            filename = ship.iloc[i,0]
            ship_label = ship.iloc[i,1]
            docked_label = docked.iloc[i,1]

            self.labels[filename] = [ship_label, docked_label]

        # Transformaciones
        if transform:
            self.transform = transform
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.ToTensor()
            ])

    # Obtener una imagen y sus etiquetas
    def __getitem__(self, index):
        image_path = self.image_files[index] # Cargamos imagen n-ésima
        image_name = os.path.splitext(os.path.basename(image_path))[0] # Nombre sin extensión
        image = read_image(image_path) # Obtenemos tensor [C, H, W]
        image = self.transform(image) # Aplicamos transformaciones
        filename = os.path.basename(image_path) # Nombre del archivo con extensión
        labels = torch.tensor(self.labels[filename], dtype=torch.float32) # Buscamos las etiquetas en el diccionario de etiquetas

        return image, labels, image_name
    
    def __len__(self):
        return len(self.image_files)

# 2.- Clasificación Ship/No-Ship

### Conjuntos de entrenamiento, validación y test

In [ ]:
# Creamos instancia del dataset
dataset = Dataset(
    image_path="C:\Users\Pc\Desktop\Uni\Tercero\segundo_cuatri\VCA\VCA\P1-Material\images",
    ship_csv="C:\Users\Pc\Desktop\Uni\Tercero\segundo_cuatri\VCA\VCA\P1-Material\docked.csv",
    docked_csv="C:\Users\Pc\Desktop\Uni\Tercero\segundo_cuatri\VCA\VCA\P1-Material\ship.csv"
)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (138900716.py, line 2)

- Obtenemos las etiquetas

In [ ]:
# Lista con las imágenes
filenames = dataset.image_files 
ship_labels = []

for f in filenames:
    filename = os.path.basename(f) # Nombre del archivo
    label = dataset.labels[filename][0] # Extraemos etiqueta
    ship_labels.append(label)

### División del dataset

El dataset se divide en tres subconjuntos: **entrenamiento**, **validación** y **test**.  
Usamos conjunto de **validación** para evaluar el rendimiento del modelo durante el entrenamiento y comparar las diferentes configuraciones (con o sin *data augmentation* o con modelos preentrenados). Reservamos el conjunto de **test** para realizar la evaluación final del modelo sobre datos no vistos.

La división se realiza de forma **estratificada** respecto a la etiqueta *ship / no-ship*, para mantener la proporción de cada clase constante en los tres subconjuntos. Esto evita sesgos en la distribución.

- Primer split: Entrenamiento y Resto

In [ ]:
train_files, temp_files, train_labels, temp_labels = train_test_split(
    filenames,
    ship_labels,
    test_size=0.30, # 70% en entrenamiento, 15% validación y 15% test
    stratify=ship_labels,
    random_state=42
)

- Segundo split: Resto en validación y test

In [ ]:
val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

- Obtenemos índices para cada conjunto

In [ ]:
# Índices para el conjunto de entrenamiento
train_idx = []

for f in train_files:
    idx = filenames.index(f)
    train_idx.append(idx)


# Índices para el conjunto de validación
val_idx = []

for f in val_files:
    idx = filenames.index(f)
    val_idx.append(idx)


# Índices para el conjunto de test
test_idx = []

for f in test_files:
    idx = filenames.index(f)
    test_idx.append(idx)

In [ ]:
train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

- Transformaciones

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

- Data Augmentation

- Creamos el dataset con Transforms

In [ ]:
dataset = Dataset(
    image_path="RUTA_IMAGES",
    ship_csv="RUTA_SHIP",
    docked_csv="RUTA_DOCKED",
    transform=transform
)

- DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

- Modelo preentrenado

In [ ]:
model = models.resnet18(pretrained=True)

# Cambiar última capa para clasificación binaria
model.fc = nn.Linear(model.fc.in_features, 1)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.train()

for epoch in range(3):

    running_loss = 0

    for images, labels, _ in train_loader:

        images = images.to(device)
        labels = labels[:,0].unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print("Epoch", epoch, "Loss:", running_loss/len(train_loader))

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels, _ in test_loader:

        images = images.to(device)
        labels = labels[:,0].unsqueeze(1).to(device)

        outputs = model(images)
        preds = torch.sigmoid(outputs) > 0.5

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Accuracy:", correct/total)